In [2]:
import surprise
import numpy as np
import pandas as pd
from surprise.model_selection import train_test_split
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [3]:
ratings = pd.read_csv(r"F:\MACHINE LEARNING\Cases_rec_sys\ml-100k\u.data", sep = '\t', names = ['uid','iid','rating','ts'])
ratings.drop('ts', axis = 1, inplace = True)
ratings.head()

,uid,iid,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [4]:
lowest_rating = ratings['rating'].min()
highest_rating=ratings['rating'].max()
lowest_rating,highest_rating

(1, 5)

In [5]:
reader = surprise.Reader(rating_scale = (lowest_rating, highest_rating))
data = surprise.Dataset.load_from_df(ratings, reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [6]:
trainset, testset = train_test_split(data, test_size= 0.25, random_state=26)
similarity_options = {'name': 'cosine', 'user_based': True}
algo = surprise.KNNBasic(sim_options = similarity_options, k=70) # default k = 40
algo.fit(trainset)
predictions = algo.test(testset)
surprise.accuracy.rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.0253


1.025282928143327

In [7]:
param_grid = {'k': [25,40,50,65,75,85], 'user_bases': [False, True]}
kfold = KFold(n_splits = 5, random_state = 26, shuffle = True)
gs = GridSearchCV(surprise.KNNWithZScore, param_grid , measures = ['rmse','mae'], cv = kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [8]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.9497919786049487
{'k': 65, 'user_bases': False}


In [9]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


In [10]:
iids = ratings['iid'].unique()
print(len(iids))

1682


In [11]:
user = 50
u_iid = ratings[ratings['uid']==user]['iid'].unique()
print('List of items rated by user: \n', u_iid)
print('No. of items rated by user {0}: {1}'.format(user,  len(u_iid)))
iids_to_predict = np.setdiff1d(iids, u_iid)
print("Items not rated by the user or those items for which the expexted ratings are to be predicted: ", iids_to_predict)

List of items rated by user: 
 [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]
No. of items rated by user 50: 24
Items not rated by the user or those items for which the expexted ratings are to be predicted:  [   1    2    3 ... 1680 1681 1682]


In [12]:
testset = [[user, iid, 0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [(predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
1164,1189,5.000000
1617,1642,5.000000
1424,1449,5.000000
1442,1467,5.000000
793,814,5.000000
1475,1500,5.000000
1511,1536,5.000000
1097,1122,5.000000
1574,1599,5.000000
1611,1636,5.000000


In [13]:
movies = pd.read_csv("F:\MACHINE LEARNING\Cases_rec_sys\ml-100k\movies_list.csv", encoding = 'latin-1')
movies.columns

Index(['movie id ', ' movie title ', ' release date ', ' video release date ',
       'IMDb URL ', ' unknown ', ' Action ', ' Adventure ', ' Animation ',
       'Children's ', ' Comedy ', ' Crime ', ' Documentary ', ' Drama ',
       ' Fantasy ', 'Film-Noir ', ' Horror ', ' Musical ', ' Mystery ',
       ' Romance ', ' Sci-Fi ', 'Thriller ', ' War ', ' Western'],
      dtype='object')

In [17]:
exp_ratings = exp_ratings.merge(movies[['movie id ', ' movie title ', ' release date ']], how = 'left', left_on= 'iid', right_on ='movie id ')
exp_ratings.sort_values(by = 'est_rating', ascending = False).head(10)

,iid,est_rating,movie id,movie title,release date
1164,1189,5.0,1189,Prefontaine (1997),24-Jan-97
1617,1642,5.0,1642,Some Mother's Son (1996),27-Dec-96
1424,1449,5.0,1449,Pather Panchali (1955),22-Mar-96
1442,1467,5.0,1467,"Saint of Fort Washington, The (1993)",01-Jan-93
793,814,5.0,814,"Great Day in Harlem, A (1994)",01-Jan-94
1475,1500,5.0,1500,Santa with Muscles (1996),08-Nov-96
1511,1536,5.0,1536,Aiqing wansui (1994),22-Jul-96
1097,1122,5.0,1122,They Made Me a Criminal (1939),01-Jan-39
1574,1599,5.0,1599,Someone Else's America (1995),10-May-96
1611,1636,5.0,1636,Brothers in Trouble (1995),26-Apr-96
